# MiniLM on Colab

**Before you run anything: Runtime -> Change runtime type -> T4 GPU -> Save.**

Everything that takes time (tokenizer, encoded data, checkpoints, logs) is written to
Google Drive, so a disconnect never costs you more than the cell you were on.


## 0. Check the GPU is actually attached


In [ ]:
import torch
print('GPU:', torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'NONE - fix the runtime type!')
print('torch:', torch.__version__)


## 1. Repo and Drive


In [ ]:
import os
if not os.path.exists('/content/OwnSLM'):
    !git clone https://github.com/AiConsultant-SiriusAI/OwnSLM.git /content/OwnSLM
%cd /content/OwnSLM
!git pull -q
!pip install -q -r requirements-dev.txt


In [ ]:
from google.colab import drive
drive.mount('/content/drive')
DATA = '/content/drive/MyDrive/minilm'
!mkdir -p {DATA}


## 2. Data (~2.2 GB, skipped if already in Drive)


In [ ]:
!wget -nc -P {DATA} https://huggingface.co/datasets/roneneldan/TinyStories/resolve/main/TinyStoriesV2-GPT4-train.txt
!wget -nc -P {DATA} https://huggingface.co/datasets/roneneldan/TinyStories/resolve/main/TinyStoriesV2-GPT4-valid.txt
!ls -lh {DATA}


## 3. Tokenizer

Runs on the CPU (Colab gives you 2 cores), so this is the slow part. `--max-bytes 550000000`
is the first quarter of the file; drop it to train on all of it. **Write down the printed
time and the longest token** - the writeup asks for both.


In [ ]:
!python -m scripts.train_tokenizer --input {DATA}/TinyStoriesV2-GPT4-train.txt --out-dir {DATA} --vocab-size 10000 --max-bytes 550000000


## 4. Encode both splits to uint16

Prints bytes/token and MB/s - two more writeup numbers.


In [ ]:
!python -m scripts.encode_data --data-dir {DATA}


## 5. Sanity check: can it memorise one batch?

The loss must fall close to zero. **If it does not, stop here** - something is wrong with the
model or the loss, and no amount of GPU time will fix it.


In [ ]:
!python train.py --train-data {DATA}/train.npy --val-data {DATA}/valid.npy --steps 300 --single-batch --eval-every 50 --ckpt-every 0 --lr 1e-3 --amp --out {DATA}/overfit


In [ ]:
!python -m scripts.plots {DATA}/overfit.csv --out overfit.png
from IPython.display import Image; Image('overfit.png')


## 6. Sanity check: resume

Run the next cell, then **Runtime -> Interrupt execution** somewhere around step 150.
Then run the cell after it, which resumes with the *same* `--steps` so the cosine schedule
matches. The joined loss curve should show no jump.


In [ ]:
!python train.py --train-data {DATA}/train.npy --val-data {DATA}/valid.npy --steps 300 --ckpt-every 50 --eval-every 100 --lr 1e-3 --amp --out {DATA}/resume


In [ ]:
!python train.py --train-data {DATA}/train.npy --val-data {DATA}/valid.npy --steps 300 --ckpt-every 50 --eval-every 100 --lr 1e-3 --amp --resume {DATA}/resume.pt --out {DATA}/resume


In [ ]:
!python -m scripts.plots {DATA}/resume.csv --out resume.png
from IPython.display import Image; Image('resume.png')


## 7. Learning-rate pilots: 5 x 300 steps

At least one rate should blow up. If none does, add a higher one.


In [ ]:
!python -m scripts.pilot_lr --steps 300 --out-dir {DATA} -- --train-data {DATA}/train.npy --val-data {DATA}/valid.npy --amp


In [ ]:
!python -m scripts.plots {DATA}/pilot_*.csv --out pilots.png
from IPython.display import Image; Image('pilots.png')


## 8. The main run

5,000 steps x batch 32 x context 256 = 40,960,000 tokens. 10-30 min on a T4.
**Set BEST_LR from the chart above** - lowest final loss, staying below anything unstable.


In [ ]:
BEST_LR = 1e-3  # <-- change me


In [ ]:
!python train.py --train-data {DATA}/train.npy --val-data {DATA}/valid.npy --steps 5000 --batch-size 32 --context-length 256 --lr {BEST_LR} --amp --ckpt-every 500 --eval-every 250 --out {DATA}/main


In [ ]:
!python -m scripts.plots {DATA}/main.csv --out loss_steps.png
!python -m scripts.plots {DATA}/main.csv --x seconds --out loss_time.png
from IPython.display import Image, display; display(Image('loss_steps.png'), Image('loss_time.png'))


## 9. Package and generate


In [ ]:
!python -m scripts.make_release --checkpoint {DATA}/main.pt --data-dir {DATA} --out release
!ls -lh release


In [ ]:
!python inference.py --model-dir release --prompt "Once upon a time" --max-new-tokens 256 --temperature 0.8 --top-p 0.9 --seed 42 --device cpu


In [ ]:
!python inference.py --model-dir release --prompt "Once upon a time" --max-new-tokens 256 --temperature 1.0 --top-p 0.95 --seed 7 --device cpu


## 10. Commit the release back to GitHub

Use a personal access token as the password when git asks (GitHub rejects account passwords).


In [ ]:
import subprocess, getpass
REPO = 'AiConsultant-SiriusAI/OwnSLM'
!git config user.email "abhijeet.chahar@siriusai.com"
!git config user.name "Abhijeet Chahar"
!cp loss_steps.png loss_time.png pilots.png overfit.png resume.png {DATA}/
!git add release && git commit -m "Add the trained model release"
# `git push` cannot prompt for a password in a notebook, so push with a token.
tok = getpass.getpass('GitHub personal access token: ')
r = subprocess.run(['git', 'push', f'https://{tok}@github.com/{REPO}.git', 'main'],
                   capture_output=True, text=True)
print((r.stdout + r.stderr).replace(tok, '***'))
